In [1]:
# library

from pathlib import Path
import numpy as np
import pandas as pd

In [3]:
#files checking
DATA_DIR = Path.cwd()

FILES = {
    "dictionary": DATA_DIR / "Data_Dictionary.xlsx",
    "organization": DATA_DIR / "Organization.xlsx",
    "food_shelf_stats": DATA_DIR / "org_FSStats.xlsx",
    "market_bucks": DATA_DIR / "marketbucksUsage.xlsx",
    "incoming_calls": DATA_DIR / "Incoming Call.xlsx",
}

inventory = pd.DataFrame(
    {
        "dataset": FILES.keys(),
        "file": [path.name for path in FILES.values()],
        "exists": [path.exists() for path in FILES.values()],
        "size_mb": [
            round(path.stat().st_size / 1_048_576, 2) if path.exists() else np.nan
            for path in FILES.values()
        ],
    }
)

display(inventory)

,dataset,file,exists,size_mb
0,dictionary,Data_Dictionary.xlsx,True,0.05
1,organization,Organization.xlsx,True,0.21
2,food_shelf_stats,org_FSStats.xlsx,True,4.18
3,market_bucks,marketbucksUsage.xlsx,True,1.04
4,incoming_calls,Incoming Call.xlsx,True,4.65


In [4]:
# check all data already upload
assert inventory["exists"].all(), "At least one required file is missing."
print("Setup complete. No source file has been changed.")


Setup complete. No source file has been changed.


In [8]:
from pathlib import Path
import subprocess
import tempfile

WORK_DIR = Path(tempfile.gettempdir()) / "minnemudac_working"
WORK_DIR.mkdir(parents=True, exist_ok=True)

WORK_FILES = {}

for name, source_path in FILES.items():
    destination = WORK_DIR / source_path.name

    command = (
        f"Copy-Item -LiteralPath '{source_path}' "
        f"-Destination '{destination}' -Force"
    )

    subprocess.run(
        ["powershell.exe", "-NoProfile", "-Command", command],
        check=True
    )

    WORK_FILES[name] = destination

print(f"Temporary working folder: {WORK_DIR}")

for name, path in WORK_FILES.items():
    print(f"{name}: {path.exists()} — {path.name}")

Temporary working folder: C:\Users\dinsa\AppData\Local\Temp\minnemudac_working
dictionary: True — Data_Dictionary.xlsx
organization: True — Organization.xlsx
food_shelf_stats: True — org_FSStats.xlsx
market_bucks: True — marketbucksUsage.xlsx
incoming_calls: True — Incoming Call.xlsx


In [9]:
# load Organization data
org_raw = pd.read_excel(
    WORK_FILES["organization"],
    sheet_name="Organization",
    dtype=object
)

# Preserve an untouched in-memory copy
org = org_raw.copy(deep=True)

print(f"Rows: {org.shape[0]:,}")
print(f"Columns: {org.shape[1]:,}")
print(f"Exact duplicate rows: {org.duplicated().sum():,}")

display(org.head())

Rows: 1,202
Columns: 68
Exact duplicate rows: 0


,__ID,CreateTime,zzCreationTS,zzModificationTS,_id_type,City,State,Zip,MailCity,MailState,...,z12MoAverage_House,z12MoAverage_Pounds,z12MoAverage_LlbsPerPerson,z12MoAverage_PeoplePerHouse,z12MoAverage_TEFAPCases,z12MoAverage_LlbsPerHH,z12MoAverage_Meals,zNameIfActiveBank,block_geoid,is_zip_centroid
0,1,System.Byte[],NaN,2026-05-15T19:15:18Z,11,Rochester,MN,55904,Rochester,MN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Channel One Regional Food Bank*,271090009031026,False
1,3,System.Byte[],NaN,2026-08-06T10:27:10Z,1,Brooklyn Park,MN,55428,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Second Harvest Heartland,270530268077009,False
2,5,System.Byte[],NaN,2026-09-01T02:10:43Z,1,East Grand Forks,MN,56721,East Grand Forks,MN,...,NaN,NaN,NaN,NaN,530,NaN,NaN,North Country Food Bank Inc.,271190202002007,False
3,6,System.Byte[],NaN,2026-09-03T10:32:32Z,1,Duluth,MN,55808,Duluth,MN,...,NaN,NaN,NaN,NaN,108,NaN,NaN,Second Harvest Northland,271370037001013,False
4,7,System.Byte[],NaN,2026-01-07T13:35:05Z,1,Fargo,ND,58102,Fargo,ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,380170006021030,False


In [10]:
# Check column names
print(f"Number of columns: {len(org.columns)}")
print(f"Duplicated column names: {org.columns.duplicated().sum()}")
print(f"Column names with outer spaces: {sum(c != c.strip() for c in org.columns)}")

display(
    pd.DataFrame({
        "column_number": range(1, len(org.columns) + 1),
        "column_name": org.columns
    })
)

Number of columns: 68
Duplicated column names: 0
Column names with outer spaces: 0


,column_number,column_name
0,1,__ID
1,2,CreateTime
2,3,zzCreationTS
3,4,zzModificationTS
4,5,_id_type
...,...,...
63,64,z12MoAverage_LlbsPerHH
64,65,z12MoAverage_Meals
65,66,zNameIfActiveBank
66,67,block_geoid


In [11]:
organization_id = pd.to_numeric(org["__ID"], errors="coerce")

print(f"Rows: {len(org):,}")
print(f"Missing __ID: {org['__ID'].isna().sum():,}")
print(f"Non-numeric __ID: {(org['__ID'].notna() & organization_id.isna()).sum():,}")
print(f"Duplicated __ID: {organization_id.duplicated().sum():,}")
print(f"Unique __ID: {organization_id.nunique():,}")

Rows: 1,202
Missing __ID: 0
Non-numeric __ID: 0
Duplicated __ID: 0
Unique __ID: 1,202


In [12]:
missing_audit = pd.DataFrame({
    "column": org.columns,
    "data_type": org.dtypes.astype(str).values,
    "missing_count": org.isna().sum().values,
    "missing_percent": (org.isna().mean().values * 100).round(2),
    "unique_nonmissing": [
        org[column].nunique(dropna=True)
        for column in org.columns
    ]
})

missing_audit = missing_audit.sort_values(
    ["missing_percent", "column"],
    ascending=[False, True]
).reset_index(drop=True)

display(missing_audit)

,column,data_type,missing_count,missing_percent,unique_nonmissing
0,isTEFAPEligible_Alert,object,1201,99.92,1
1,WebscmShipToName,object,1197,99.58,5
2,_id_IfFoodBank,object,1197,99.58,5
3,isWebscmShipTo,object,1197,99.58,1
4,zNameIfActiveBank,object,1197,99.58,5
...,...,...,...,...,...
63,_id_type,object,33,2.75,9
64,isDistributionSite,object,23,1.91,2
65,ActiveInactive,object,8,0.67,2
66,__ID,object,0,0.00,1202


In [13]:
fully_empty_columns = missing_audit.loc[
    missing_audit["missing_percent"] == 100,
    "column"
].tolist()

complete_columns = missing_audit.loc[
    missing_audit["missing_percent"] == 0,
    "column"
].tolist()

print("Fully empty columns:")
print(fully_empty_columns)

print("\nColumns with no missing values:")
print(complete_columns)

Fully empty columns:
[]

Columns with no missing values:
['__ID', 'zzModificationTS']


In [14]:
def audit_text_column(series):
    text_mask = series.map(lambda value: isinstance(value, str))
    text = series.loc[text_mask]

    return pd.Series({
        "text_values": len(text),
        "blank_strings": text.str.strip().eq("").sum(),
        "outer_whitespace": text.ne(text.str.strip()).sum(),
        "line_breaks": text.str.contains(r"[\r\n]", regex=True).sum(),
        "repeated_spaces": text.str.contains(r" {2,}", regex=True).sum(),
    })


text_audit = (
    org.apply(audit_text_column)
       .T
       .reset_index(names="column")
)

text_issues = text_audit.loc[
    text_audit[
        ["blank_strings", "outer_whitespace", "line_breaks", "repeated_spaces"]
    ].sum(axis=1) > 0
].reset_index(drop=True)

display(text_issues)

,column,text_values,blank_strings,outer_whitespace,line_breaks,repeated_spaces
0,City,1061,0,31,0,0
1,State,1059,0,1,1,0
2,MailCity,909,0,23,0,0
3,MailState,909,0,2,0,0
4,Eligibility,19,0,0,0,1
5,Conditions,19,0,4,0,2
6,MarketSeasons,60,0,0,20,0
7,AgencyType,1169,0,6,0,0


In [15]:
for column in text_issues["column"]:
    problem_mask = org[column].map(
        lambda value:
            isinstance(value, str)
            and (
                value != value.strip()
                or value.strip() == ""
                or "\n" in value
                or "\r" in value
                or "  " in value
            )
    )

    print(f"\n--- {column} ---")
    display(
        org.loc[problem_mask, ["__ID", column]]
           .head(20)
    )


--- City ---


,__ID,City
23,1088,Duluth
61,1143,Cannon Falls
62,1144,Pillager
84,1172,Baudette
106,1202,Bethel
113,1214,Paynesville
131,1233,Olivia
245,13879,Young America
265,14091,Janesville
306,14181,Barnesville



--- State ---


,__ID,State
757,15677,MN\n



--- MailCity ---


...

,__ID,Conditions
45,1116,You may use the Food Pantry twice a month and...
606,14739,Clients able access once a week.
951,16703,You may use the Food Pantry twice a month and...
1125,16885,Clients able access once a week.



--- MarketSeasons ---


,__ID,MarketSeasons
323,14240,Winter\nSummer
329,14246,Winter\nSummer
330,14247,Winter\nSummer
333,14250,Winter\nSummer
335,14252,Winter\nSummer
336,14253,Winter\nSummer
345,14263,Winter\nSummer
355,14273,Winter\nSummer
356,14274,Winter\nSummer
358,14276,Winter\nSummer



--- AgencyType ---


,__ID,AgencyType
44,1113,College Campus Program
304,14177,College Campus Program
626,14793,College Campus Program
693,15212,College Campus Program
716,15310,College Campus Program
889,16393,College Campus Program


In [16]:
# Restart from the untouched source so this cell is reproducible
org = org_raw.copy(deep=True)

cleaning_log = []

for column in org.columns:
    is_text = org[column].map(lambda value: isinstance(value, str))

    stripped = org.loc[is_text, column].str.strip()

    blank_index = stripped.index[stripped.eq("")]
    trim_index = stripped.index[
        stripped.ne("")
        & org.loc[is_text, column].ne(stripped)
    ]

    if len(blank_index) > 0:
        org.loc[blank_index, column] = pd.NA

    if len(trim_index) > 0:
        org.loc[trim_index, column] = stripped.loc[trim_index]

    if len(blank_index) > 0 or len(trim_index) > 0:
        cleaning_log.append({
            "dataset": "Organization",
            "column": column,
            "rule": "Trim outer whitespace and convert blank strings to missing",
            "trimmed_values": len(trim_index),
            "blank_strings_replaced": len(blank_index),
        })

cleaning_log = pd.DataFrame(cleaning_log)

display(cleaning_log)

,dataset,column,rule,trimmed_values,blank_strings_replaced
0,Organization,City,Trim outer whitespace and convert blank string...,31,0
1,Organization,State,Trim outer whitespace and convert blank string...,1,0
2,Organization,MailCity,Trim outer whitespace and convert blank string...,23,0
3,Organization,MailState,Trim outer whitespace and convert blank string...,2,0
4,Organization,Conditions,Trim outer whitespace and convert blank string...,4,0
5,Organization,AgencyType,Trim outer whitespace and convert blank string...,6,0


In [17]:
remaining_blank_strings = 0
remaining_outer_whitespace = 0

for column in org.columns:
    is_text = org[column].map(lambda value: isinstance(value, str))
    text = org.loc[is_text, column]

    remaining_blank_strings += text.str.strip().eq("").sum()
    remaining_outer_whitespace += text.ne(text.str.strip()).sum()

print(f"Remaining blank strings: {remaining_blank_strings}")
print(f"Remaining outer whitespace: {remaining_outer_whitespace}")

Remaining blank strings: 0
Remaining outer whitespace: 0


## Step 7: Audit State and ZIP fields before correcting them

In [18]:
address_columns = [
    "City", "State", "Zip",
    "MailCity", "MailState", "MailZip"
]

display(org[address_columns].head(10))

,City,State,Zip,MailCity,MailState,MailZip
0,Rochester,MN,55904,Rochester,MN,55904
1,Brooklyn Park,MN,55428,NaN,NaN,NaN
2,East Grand Forks,MN,56721,East Grand Forks,MN,56721
3,Duluth,MN,55808,Duluth,MN,55808
4,Fargo,ND,58102,Fargo,ND,58102
5,Anoka,MN,55303,Anoka,MN,55303
6,Albany,MN,56307,Albany,MN,56307
7,Minneapolis,MN,55409,Minneapolis,MN,55409
8,Rice,MN,56367,Rice,MN,56367
9,Annandale,MN,55302,Annandale,MN,55302


In [19]:
for column in ["State", "MailState"]:
    print(f"\n--- {column} ---")
    display(
        org[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="rows")
    )


--- State ---


,State,rows
0,MN,1058
1,NaN,140
2,ND,1
3,55313,1
4,55056,1
5,55318,1



--- MailState ---


,MailState,rows
0,MN,907
1,NaN,293
2,ND,2


In [20]:
def valid_state_abbreviation(value):
    if pd.isna(value):
        return True

    return bool(
        pd.Series([str(value).upper()])
        .str.fullmatch(r"[A-Z]{2}")
        .iloc[0]
    )


invalid_state_mask = ~org["State"].map(valid_state_abbreviation)

invalid_state_rows = org.loc[
    invalid_state_mask,
    ["__ID", "City", "State", "Zip"]
]

display(invalid_state_rows)

,__ID,City,State,Zip
1185,16969,Buffalo,55313,NaN
1188,16972,North Branch,55056,NaN
1189,16974,Chaska,55318,NaN
